# Phase 3 Project: SyriaTel Customer Churn Classification

**Author:** [Your Name]  
**Date:** 2024  
**Dataset:** [SyriaTel Customer Churn — Kaggle](https://www.kaggle.com/datasets/becksddf/churn-in-telecoms-dataset)  
**Repository:** [Your GitHub Link Here]  

---

## 1. Business Understanding

### The Problem
Customer churn is one of the most costly problems in the telecom industry. It costs SyriaTel 5-10x more to acquire a new customer than to retain an existing one. Every churned customer represents lost recurring revenue.

### Stakeholder
**SyriaTel's Customer Retention Team.** This team reduces churn by identifying at-risk customers and deploying targeted interventions (discounts, outreach calls, service upgrades) before customers cancel.

### How They Would Use This Project
The retention team would run this model monthly against their full customer database. Customers flagged as high churn risk are prioritized for proactive outreach. Without a model, the team operates reactively and only learns of dissatisfaction after revenue is already lost.

### Why Machine Learning, Not Simple Rules?
Churn is driven by a complex combination of usage, billing, complaints, and plan type all at once. A simple threshold rule (e.g. 4+ service calls) misses customers who churn for different reasons. A classification model finds non-obvious multi-variable patterns humans would miss.

### Why Classification, Not Regression?
Churn is binary: a customer either churns (1) or stays (0). We are predicting group membership, not a numeric value. Classification is the correct tool.

### Metric Choice: Recall
We use **recall** as our primary metric. It answers: *of all customers who actually churned, what fraction did our model correctly catch?*

- A **false negative** = missed churner → they leave with no intervention → revenue permanently lost  
- A **false positive** = loyal customer flagged → unnecessary outreach → small wasted cost  

Missing a churner is far more costly than flagging a loyal customer, so maximizing recall is the correct business priority. We also report accuracy and F1 for completeness.

## 2. Data Understanding

### Source
**Dataset:** SyriaTel Customer Churn  
**URL:** https://www.kaggle.com/datasets/becksddf/churn-in-telecoms-dataset  
**File:** `bigml_59c28831336c6604c800002a.csv` — 3,333 rows, 21 columns. Each row is one customer account. Collected by BigML.

This dataset is well-suited for this project because it captures the exact variables a telecom company tracks — usage, billing, complaints, plan type — and includes actual churn outcomes as the target.

### Data Limitations
- **Single time snapshot:** Cannot capture trends in individual customer behavior over time
- **No competitor data:** Competitor pricing or offers are not captured
- **No network quality data:** Signal strength and outage history are absent
- **Limited geography:** Only state-level location is available

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay, recall_score
)
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

# Load data
# Download from: https://www.kaggle.com/datasets/becksddf/churn-in-telecoms-dataset
# Place bigml_59c28831336c6604c800002a.csv in the same folder as this notebook
df = pd.read_csv('bigml_59c28831336c6604c800002a.csv')

print('Shape:', df.shape)
print('\nTarget distribution:')
print(df['churn'].value_counts())
print(f'\nChurn rate: {df["churn"].mean():.1%}')
df.head()

In [ ]:
# Data quality and descriptive statistics
print('Missing values per column:')
print(df.isnull().sum())
print('\nDescriptive statistics for numeric features:')
df.describe().round(2)

### Feature Relevance Assessment

| Feature | Type | Include? | Justification |
|---|---|---|---|
| `phone number` | string ID | **Drop** | Unique identifier — would cause data leakage |
| `state` | categorical (51 values) | **Drop** | Too many levels; no direct churn signal |
| `area code` | numeric ID | **Drop** | Identifier, not predictive |
| `account length` | numeric | Keep | Tenure may correlate with loyalty |
| `international plan` | yes/no | Keep | Known strong churn driver |
| `voice mail plan` | yes/no | Keep | Indicates feature engagement |
| `number vmail messages` | numeric | Keep | Engagement indicator |
| `total day/eve/night minutes` | numeric | Keep | Usage volume drives charges |
| `total day/eve/night charge` | numeric | Keep | High charges = price sensitivity = churn risk |
| `total intl minutes/calls/charge` | numeric | Keep | International usage and cost |
| `customer service calls` | numeric | Keep | Strongest known churn predictor |

In [ ]:
# Class distribution and key feature exploration
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['churn'].value_counts().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'coral']
)
axes[0].set_title('Churn Distribution (~14.5% churn rate — imbalanced)')
axes[0].set_xticklabels(['No Churn', 'Churn'], rotation=0)
axes[0].set_ylabel('Count')

df.groupby('churn')['customer service calls'].mean().plot(
    kind='bar', ax=axes[1], color=['steelblue', 'coral']
)
axes[1].set_title('Avg Customer Service Calls by Churn')
axes[1].set_xticklabels(['No Churn', 'Churn'], rotation=0)
axes[1].set_ylabel('Average Calls')

plt.tight_layout()
plt.show()

print('The dataset is imbalanced (~14.5% churn rate).')
print('A naive model predicting no-churn for everyone scores 85.5% accuracy but catches 0 churners.')
print('This confirms recall is the right metric and class_weight handling is required.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df.groupby('international plan')['churn'].mean().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'coral']
)
axes[0].set_title('Churn Rate by International Plan')
axes[0].set_xticklabels(['No Intl Plan', 'Has Intl Plan'], rotation=0)
axes[0].set_ylabel('Churn Rate')

df.boxplot(column='total day charge', by='churn', ax=axes[1])
axes[1].set_title('Total Day Charge by Churn Status')
axes[1].set_xlabel('Churn (0 = No, 1 = Yes)')
plt.suptitle('')

plt.tight_layout()
plt.show()

print('International plan customers churn at ~3x the rate of non-international customers.')
print('Higher day charges correlate with churn — price-sensitive customers are more likely to switch.')

## 3. Data Preparation

### Steps and Justifications

1. **Drop `phone number`, `state`, `area code`:** Identifiers with no predictive signal. `phone number` would cause leakage as it uniquely tags rows.
2. **LabelEncode `international plan` and `voice mail plan`:** Yes/no strings converted to 0/1 for sklearn compatibility.
3. **Convert `churn` to integer:** Boolean True/False converted to 1/0 for classifier and metric compatibility.
4. **Stratified 80/20 split:** Stratification preserves the 14.5% churn rate identically in train and test sets.
5. **Scaling inside `evaluate()`:** Scaler is always fit on training data only and used transform-only on test — preventing leakage.

In [ ]:
# Step 1: Drop identifier columns
df = df.drop(columns=['phone number', 'state', 'area code'])

# Step 2: Encode yes/no columns to 0/1
le = LabelEncoder()
df['international plan'] = le.fit_transform(df['international plan'])
df['voice mail plan']    = le.fit_transform(df['voice mail plan'])

# Step 3: Convert churn boolean to integer
df['churn'] = df['churn'].astype(int)

# Step 4: Separate features and target
X = df.drop('churn', axis=1)
y = df['churn']

# Step 5: Stratified train-test split — preserves churn ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', X_train.shape[0], ' | Test size:', X_test.shape[0])
print('Train churn rate:', round(y_train.mean(), 3))
print('Test  churn rate:', round(y_test.mean(),  3))
print('Churn rates match — stratification confirmed.')
print('\nFinal feature list:', X.columns.tolist())

In [ ]:
def evaluate(model, name, needs_scaling=False):
    """
    Evaluate a fitted sklearn model on train and test sets.

    Parameters
    ----------
    model         : fitted sklearn estimator
    name          : str, label for output
    needs_scaling : bool
        True  — distance-based models (logistic regression).
                 Scaler fit on X_train only, transform-only on X_test (no leakage).
        False — tree-based models or Pipelines that handle scaling internally.
    """
    if needs_scaling:
        s = StandardScaler()
        Xtr = s.fit_transform(X_train)  # fit on training data only
        Xte = s.transform(X_test)       # transform only — never fit on test
    else:
        Xtr, Xte = X_train, X_test

    train_recall = recall_score(y_train, model.predict(Xtr))
    test_recall  = recall_score(y_test,  model.predict(Xte))
    train_acc    = model.score(Xtr, y_train)
    test_acc     = model.score(Xte, y_test)

    print(f"\n{'='*50}")
    print(f"Model: {name}")
    print(f"Train Accuracy: {train_acc:.3f}  |  Train Recall: {train_recall:.3f}")
    print(f"Test  Accuracy: {test_acc:.3f}  |  Test  Recall: {test_recall:.3f}")
    print("\nTest Classification Report:")
    print(classification_report(
        y_test, model.predict(Xte), target_names=['No Churn', 'Churn']
    ))

## 4. Modeling

Iterative approach — each model is justified by the results of the prior one:

| # | Model | Why we build it |
|---|---|---|
| 1 | Logistic Regression | Simple interpretable baseline |
| 2 | Decision Tree (default) | Nonparametric; checks for non-linear patterns and overfitting |
| 3 | Decision Tree (tuned) | Addresses overfitting found in Model 2; improves recall |
| 4 | Random Forest + Pipeline | Ensemble reduces variance; Pipeline enforces best-practice leakage prevention |

### Model 1: Logistic Regression (Baseline)

In [ ]:
# Distance-based model — requires scaling; fit scaler on train only
s = StandardScaler()
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(s.fit_transform(X_train), y_train)

evaluate(log_reg, 'Logistic Regression (Baseline)', needs_scaling=True)

**Observation & Justification for Next Step:**  
Logistic regression achieves solid accuracy but very low recall on churners. Two reasons: (1) class imbalance biases it toward predicting the majority class; (2) a linear boundary cannot capture non-linear interactions between usage variables and churn. A nonparametric decision tree should handle both issues better.

### Model 2: Decision Tree (Default)

In [ ]:
# Trees do not require feature scaling
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

evaluate(dt, 'Decision Tree (Default)', needs_scaling=False)

**Observation & Justification for Next Step:**  
The default tree hits perfect train accuracy (1.000) but lower test accuracy — clear **overfitting**. An unconstrained tree memorizes every training example including noise. We tune `max_depth` and `min_samples_leaf` to constrain complexity. We also add `class_weight='balanced'` to address the class imbalance.

### Model 3: Tuned Decision Tree (GridSearchCV)

In [ ]:
param_grid = {
    'max_depth':        [3, 5, 7, 10, None],
    'min_samples_leaf': [1, 2, 5, 10],
    'class_weight':     [None, 'balanced']  # 'balanced' upweights minority churn class
}

# Score on recall — optimize for catching churners, not just overall accuracy
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid, cv=5, scoring='recall'
)
grid_search.fit(X_train, y_train)

best_dt = grid_search.best_estimator_
print('Best hyperparameters:', grid_search.best_params_)

evaluate(best_dt, 'Decision Tree (Tuned)', needs_scaling=False)

**Observation & Justification for Next Step:**  
Tuning significantly improves recall and reduces the train/test gap, confirming overfitting was addressed. However, a single decision tree still has high variance. A Random Forest averages many trees to reduce variance while preserving non-linear pattern recognition. We wrap it in a Pipeline for best-practice architecture.

### Model 4: Random Forest with Pipeline (Final Model)

In [ ]:
# Pipeline: preprocessing and model bundled — structurally prevents data leakage
# When .fit() is called, scaler only sees training data
# When .predict() is called, same fitted scaler transforms new data automatically
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',  # addresses 14.5% class imbalance
        random_state=42
    ))
])

pipe.fit(X_train, y_train)

# needs_scaling=False — Pipeline handles scaling internally
evaluate(pipe, 'Random Forest (Pipeline)', needs_scaling=False)

**Improvement:** Random Forest achieves higher test recall than the tuned decision tree, confirming ensemble averaging reduces variance and yields more reliable predictions on unseen data.

## 5. Evaluation

### Final Model Selection
We select the **Random Forest Pipeline** as our final model based on its superior test recall — the metric that directly captures business value for this churn prediction problem.

In [ ]:
# Confusion matrix on holdout test set
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(
    pipe, X_test, y_test,
    display_labels=['No Churn', 'Churn'], ax=ax, colorbar=False
)
ax.set_title('Random Forest — Confusion Matrix (Holdout Test Set)')
plt.tight_layout()
plt.show()

print('Bottom-left = False Negatives: churners missed — most costly error for SyriaTel')
print('Top-right   = False Positives: loyal customers flagged — less costly')

In [ ]:
# Feature importances — which variables the model relies on most
importances = pd.Series(
    pipe.named_steps['rf'].feature_importances_,
    index=X.columns
).sort_values(ascending=True)

importances.plot(kind='barh', figsize=(8, 7), color='steelblue')
plt.title('Feature Importances — Random Forest Final Model')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 5 churn predictors:')
print(importances.sort_values(ascending=False).head(5))

In [ ]:
# Model comparison — all evaluated on holdout test set
s_compare = StandardScaler()
Xtr_s = s_compare.fit_transform(X_train)
Xte_s = s_compare.transform(X_test)

model_dict = {
    'Logistic Regression':      (log_reg, Xtr_s,  Xte_s),
    'Decision Tree (Default)':  (dt,      X_train, X_test),
    'Decision Tree (Tuned)':    (best_dt, X_train, X_test),
    'Random Forest (Pipeline)': (pipe,    X_train, X_test)
}

results = []
for name, (m, Xtr, Xte) in model_dict.items():
    results.append({
        'Model':       name,
        'Train Acc':   round(m.score(Xtr, y_train), 3),
        'Test Acc':    round(m.score(Xte, y_test),  3),
        'Test Recall': round(recall_score(y_test, m.predict(Xte)), 3)
    })

pd.DataFrame(results)

### Implications for SyriaTel

The Random Forest Pipeline correctly identifies the majority of actual churners on the holdout test set. In practical terms:

- The model gives the retention team an actionable churn risk score for every customer each month
- Customers flagged can receive proactive retention incentives before they cancel
- Some false positives will occur — SyriaTel should calculate the break-even point (outreach cost vs. lifetime value of a retained customer) to set the right classification threshold
- The model is most reliable for customers with the top predictors: high day charges, multiple service calls, and international plan

## 6. Conclusion

### Summary
We built a churn classification model using SyriaTel account-level data. After iterating through four models, the **Random Forest Pipeline** was selected as the final model based on its best test recall — meaning it catches the highest fraction of actual churners from unseen holdout data.

### Key Findings
The strongest churn predictors were **total day charge**, **customer service calls**, and **international plan status**:
- High daily charges indicate price-sensitive customers likely to switch to cheaper alternatives
- Frequent service calls indicate unresolved problems eroding satisfaction
- International plan customers churn at ~3x the average rate, suggesting unmet specialized needs

### Recommendations for SyriaTel
1. **Run this model monthly** to score all active customers and surface the highest-risk accounts
2. **Target proactive outreach** at customers with both high day charges and 3+ service calls
3. **Review international plan pricing immediately** — ~3x churn rate signals a pricing or service gap
4. **Track 90-day outcomes** after interventions and retrain the model quarterly

### Limitations
- **Time snapshot:** Churn patterns shift with market changes; regular retraining is required
- **Precision trade-off:** Higher recall means some outreach targets customers who would not have churned; weigh outreach cost vs. retention value before full deployment
- **Missing data:** Competitor offers, network quality, and satisfaction scores would likely improve model performance
- **Validation needed:** Test on a fresh customer cohort before full production deployment